## CONEXAO COM O BANCO DE DADOS

In [1]:
import os

from sqlalchemy import create_engine,text
from sqlalchemy.engine import URL

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=os.environ["USER"],
    password=os.environ["PWD"],
    host="localhost",
    port=3306,
    database="enem"
)

engine = create_engine(connection_url)

# Testar conexão com o banco
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print(result.fetchone())

(1,)


## ETL

### EXTRACT AND TRANSFORM

#### TABELAS DE DADOS GEOGRAFICOS

In [2]:
import pandas as pd

# Usei a de 2023 pois é a primeira que adota a nova divisão do IBGE que consiste em Regiões Geográficas Imediatas e Regiões Geográficas Intermediárias

censo2023 = pd.read_csv(r"E:\fbd\microdados_censo_escolar_2023\microdados_censo_escolar_2023\dados\microdados_ed_basica_2023.csv", encoding = "latin1", sep = ";")

# Filtrar atributos de interesse e remover duplicatas
# Cada registro da tabela censo2023_dedup referencia um municipio

censo2023_dedup = censo2023.iloc[:,1:12].drop_duplicates().reset_index(drop = True)

# Criar tabela REGIAO
# Filtrar atributos de interesse e remover duplicatas

REGIAO = censo2023_dedup.iloc[:,:2].drop_duplicates().reset_index(drop=True) 

REGIAO = REGIAO[["CO_REGIAO","NO_REGIAO"]] # Reordenar colunas para que a primary key seja o primeiro atributo

# Criar tabela UNIDADE_FEDERATIVA
# Filtrar atributos de interesse e remover duplicatas

UNIDADE_FEDERATIVA = censo2023_dedup.iloc[:,1:5].drop_duplicates().reset_index(drop=True)

UNIDADE_FEDERATIVA = UNIDADE_FEDERATIVA[["CO_UF","SG_UF","NO_UF", "CO_REGIAO"]] # Reordenar colunas

# Criar tabela REGIAO_GEOGRAFICA_INTERMEDIARIA
# Filtrar atributos de interesse e remover duplicatas

REGIAO_GEOGRAFICA_INTERMEDIARIA = censo2023_dedup.iloc[:,[4, 7, 8]].drop_duplicates().reset_index(drop=True)

REGIAO_GEOGRAFICA_INTERMEDIARIA = REGIAO_GEOGRAFICA_INTERMEDIARIA[["CO_REGIAO_GEOG_INTERM","NO_REGIAO_GEOG_INTERM", "CO_UF"]] # Reordenar colunas

# Criar tabela REGIAO_GEOGRAFICA_IMEDIATA
# Filtrar atributos de interesse e remover duplicatas

REGIAO_GEOGRAFICA_IMEDIATA = censo2023_dedup.iloc[:,8:].drop_duplicates().reset_index(drop = True)

REGIAO_GEOGRAFICA_IMEDIATA = REGIAO_GEOGRAFICA_IMEDIATA[["CO_REGIAO_GEOG_IMED", "NO_REGIAO_GEOG_IMED", "CO_REGIAO_GEOG_INTERM"]] # Reordenar colunas

# Criar tabela MUNICIPIO
# Filtrar atributos de interesse e remover duplicatas

MUNICIPIO = censo2023_dedup.iloc[:,[5,6,10]].drop_duplicates().reset_index(drop=True)

MUNICIPIO = MUNICIPIO[["CO_MUNICIPIO","NO_MUNICIPIO","CO_REGIAO_GEOG_IMED"]] # Reordenar colunas

# Renomear atributos para que seus nomes nao fiquem muito extensos

MUNICIPIO = MUNICIPIO.rename(columns = {"CO_REGIAO_GEOG_IMED" : "CO_RG_IMED"})

REGIAO_GEOGRAFICA_IMEDIATA = REGIAO_GEOGRAFICA_IMEDIATA.rename(columns = {"CO_REGIAO_GEOG_IMED": "CO_RG_IMED", "NO_REGIAO_GEOG_IMED": "NO_RG_IMED", "CO_REGIAO_GEOG_INTERM": "CO_RG_INTERMED"})

REGIAO_GEOGRAFICA_INTERMEDIARIA = REGIAO_GEOGRAFICA_INTERMEDIARIA.rename(columns = {"CO_REGIAO_GEOG_INTERM" : "CO_RG_INTERMED", "NO_REGIAO_GEOG_INTERM" : "NO_RG_INTERMED"})

C:\Users\segur\AppData\Local\Temp\ipykernel_15336\2468413631.py:5: DtypeWarning: Columns (0: CO_ORGAO_REGIONAL) have mixed types. Specify dtype option on import or set low_memory=False.
  censo2023 = pd.read_csv(r"E:\fbd\microdados_censo_escolar_2023\microdados_censo_escolar_2023\dados\microdados_ed_basica_2023.csv", encoding = "latin1", sep = ";")


#### TABELAS DE DADOS DO ENEM

In [3]:
import pandas as pd

####### PREPARAÇÕES INICIAIS DOS DADOS #######

# Leitura das planilhas do INEP

participantes2019 = pd.read_csv(r"E:\fbd\microdados_enem_2019\DADOS\MICRODADOS_ENEM_2019.csv", encoding = "latin1", sep = ";")
participantes2020 = pd.read_csv(r"E:\fbd\microdados_enem_2020\DADOS\MICRODADOS_ENEM_2020.csv", encoding = "latin1", sep = ";")
participantes2021 = pd.read_csv(r"E:\fbd\microdados_enem_2021\DADOS\MICRODADOS_ENEM_2021.csv", encoding = "latin1", sep = ";")
participantes2022 = pd.read_csv(r"E:\fbd\microdados_enem_2022\DADOS\MICRODADOS_ENEM_2022.csv", encoding = "latin1", sep = ";")
participantes2023 = pd.read_csv(r"E:\fbd\microdados_enem_2023\microdados_enem_2023\DADOS\MICRODADOS_ENEM_2023.csv", encoding = "latin1", sep = ";")

# Identificação e remoção de colunas irrelevantes

id_col_lixo = [8, 9, 10, 13, 14, 16, 17, 18, 20, 21, 22, 35, 36, 37, 38, 40, 41, 42, 43]

participantes2019 = participantes2019.drop(participantes2019.columns[id_col_lixo],axis = 1)
participantes2020 = participantes2020.drop(participantes2020.columns[id_col_lixo],axis = 1)
participantes2021 = participantes2021.drop(participantes2021.columns[id_col_lixo],axis = 1)
participantes2022 = participantes2022.drop(participantes2022.columns[id_col_lixo],axis = 1)
participantes2023 = participantes2023.drop(participantes2023.columns[id_col_lixo],axis = 1)

# Remoção de registros com valores ausentes

participantes2019_clean = participantes2019.dropna()
participantes2020_clean = participantes2020.dropna()
participantes2021_clean = participantes2021.dropna()
participantes2022_clean = participantes2022.dropna()
participantes2023_clean = participantes2023.dropna()

# Estratificação de cada tabela baseado na Unidade Federativa na qual cada participante estuda
# Em seguida, seleciona-se aleatoriamente 200 registros de cada UF

p2019_strat = participantes2019_clean.groupby("SG_UF_ESC", group_keys = False).sample(n = 200, random_state = 42).reset_index(drop = True)
p2020_strat = participantes2020_clean.groupby("SG_UF_ESC", group_keys = False).sample(n = 200, random_state = 42).reset_index(drop = True)
p2021_strat = participantes2021_clean.groupby("SG_UF_ESC", group_keys = False).sample(n = 200, random_state = 42).reset_index(drop = True)
p2022_strat = participantes2022_clean.groupby("SG_UF_ESC", group_keys = False).sample(n = 200, random_state = 42).reset_index(drop = True)
p2023_strat = participantes2023_clean.groupby("SG_UF_ESC", group_keys = False).sample(n = 200, random_state = 42).reset_index(drop = True)

# Concatenação das tabelas estratificadas somando 27000 registros (1000 por UF)

participantes_completo = pd.concat([p2019_strat,p2020_strat,p2021_strat,p2022_strat,p2023_strat], ignore_index = True)

# remoção de coluna referente à sigla da UF que nao será mais utilizada

participantes_completo = participantes_completo.drop(columns=["SG_UF_ESC"])

####### CRIAÇÃO DAS TABELAS RELATIVAS AO QUESTIONÁRIO SOCIOECONÔMICO ####### 

# 1) TABELA QUESTAO_QUESTIONARIO

# Leitura da planilha contendo as descrições de cada questão do questionário socioeconômico

QUESTAO_QUESTIONARIO = pd.read_csv(r"E:\fbd\questionariosocio.csv", encoding="utf-8", sep = ";", header = None, names = ["DESCRICAO"])

# Definição do código das questões

codigo_questoes = [f"Q{i:03d}" for i in range(1,26)]

# Inserção do código das questões na tabela e reordenação das colunas

QUESTAO_QUESTIONARIO["COD_QUESTAO"] = codigo_questoes
QUESTAO_QUESTIONARIO = QUESTAO_QUESTIONARIO[["COD_QUESTAO", "DESCRICAO"]]

# 2) TABELA RESPOSTA_QUESTIONARIO

# Extrair somente o conteúdo referente ao questionário, além do NU_INSCRICAO que fará parte da chave composta

respostas = participantes_completo.drop(participantes_completo.columns[1:31], axis = 1)

# Tornar cada resposta às questões em um único registro
# E converter o nome das colunas referente às questões em dados para formar a chave composta em conjunto com NU_INSCRICAO

RESPOSTA_QUESTIONARIO = respostas.melt(
    id_vars = "NU_INSCRICAO",
    value_vars = codigo_questoes,
    var_name = "COD_QUESTAO",
    value_name = "RESPOSTA"
)

# Converter os dados em string e padronizar os valores

RESPOSTA_QUESTIONARIO["RESPOSTA"] = RESPOSTA_QUESTIONARIO["RESPOSTA"].astype(str)

RESPOSTA_QUESTIONARIO["RESPOSTA"] = RESPOSTA_QUESTIONARIO["RESPOSTA"].str[:2]

RESPOSTA_QUESTIONARIO["RESPOSTA"] = RESPOSTA_QUESTIONARIO["RESPOSTA"].str.replace(".","", regex = False)

# Criar tabela EDICAO_ENEM
# Selecionar apenas os anos do ENEM

EDICAO_ENEM = participantes_completo[["NU_ANO"]].drop_duplicates().reset_index(drop = True)

EDICAO_ENEM = EDICAO_ENEM.rename(columns = {"NU_ANO": "EDICAO"})

# Criação de chave artificial para a entidade RESULTADO_REDACAO

participantes_completo["ID_RESULTADO_RED"] = participantes_completo.index + 1

# Criar tabela PARTICIPANTE
# Remoção de colunas irrelevantes para a tabela

PARTICIPANTE = participantes_completo.drop(participantes_completo.columns[11:56], axis = 1)

# Mover coluna referente ao ano do Enem para o final da tabela

nu_ano = PARTICIPANTE.pop("NU_ANO")

PARTICIPANTE["NU_ANO"] = nu_ano

# Alterar tipo do dado da coluna CO_MUNICIPIO_ESC de float para int

PARTICIPANTE["CO_MUNICIPIO_ESC"] = PARTICIPANTE["CO_MUNICIPIO_ESC"].astype(int)

# Remover coluna ID_RESULTADO_RED da tabela PARTICIPANTE

PARTICIPANTE.pop("ID_RESULTADO_RED")

# Realocar coluna TP_SEXO

tp_sexo = PARTICIPANTE.pop("TP_SEXO")

PARTICIPANTE.insert(1, "TP_SEXO", tp_sexo)

# Renomear coluna referente ao municipio onde o participante estuda

PARTICIPANTE = PARTICIPANTE.rename(columns = {"CO_MUNICIPIO_ESC" : "CO_MUNICIPIO_ESCOLA", "NU_ANO" : "ANO_ENEM"})

# Criar tabela RESULTADO_REDACAO
# Remoção das colunas irrelevantes

RESULTADO_REDACAO = participantes_completo.drop(participantes_completo.columns[1:24],axis = 1)

RESULTADO_REDACAO = RESULTADO_REDACAO.drop(RESULTADO_REDACAO.columns[8:33],axis = 1)

# Inserção da coluna ID_RESULTADO_RED na primeira posição da tabela

id_resultado_red = RESULTADO_REDACAO.pop("ID_RESULTADO_RED")

RESULTADO_REDACAO.insert(0, "ID_RESULTADO_RED", id_resultado_red)

# Conversão do tipo de dado de float para int da coluna TP_STATUS_REDACAO

RESULTADO_REDACAO["TP_STATUS_REDACAO"] = RESULTADO_REDACAO["TP_STATUS_REDACAO"].astype(int)

# Criar tabela RESULTADO_PROVA
# Remoção de colunas irrelevantes

dados_resultado = participantes_completo.drop(participantes_completo.columns[1:11],axis = 1)
dados_resultado = dados_resultado.drop(dados_resultado.columns[14:], axis = 1)

# Separação dos dados de cada prova e padronização da nomenclatura referente à nota e à presença

cn = dados_resultado[["NU_INSCRICAO","CO_PROVA_CN", "TP_PRESENCA_CN", "NU_NOTA_CN"]].rename(
    columns = {"CO_PROVA_CN" : "CO_PROVA", "TP_PRESENCA_CN" : "TP_PRESENCA", "NU_NOTA_CN" : "NU_NOTA"})

ch = dados_resultado[["NU_INSCRICAO","CO_PROVA_CH", "TP_PRESENCA_CH", "NU_NOTA_CH"]].rename(
    columns = {"CO_PROVA_CH" : "CO_PROVA", "TP_PRESENCA_CH" : "TP_PRESENCA", "NU_NOTA_CH" : "NU_NOTA"})

# Note que a prova de linguagens possui o atributo de lingua estrangeira tambem

lc = dados_resultado[["NU_INSCRICAO","CO_PROVA_LC", "TP_PRESENCA_LC", "NU_NOTA_LC", "TP_LINGUA"]].rename(
    columns = {"CO_PROVA_LC" : "CO_PROVA", "TP_PRESENCA_LC" : "TP_PRESENCA", "NU_NOTA_LC" : "NU_NOTA"})

mt = dados_resultado[["NU_INSCRICAO","CO_PROVA_MT", "TP_PRESENCA_MT", "NU_NOTA_MT"]].rename(
    columns = {"CO_PROVA_MT" : "CO_PROVA", "TP_PRESENCA_MT" : "TP_PRESENCA", "NU_NOTA_MT" : "NU_NOTA"})

# Adição da coluna de lingua estrangeira que ficará com valores Null para as outras provas

provas_sem_lingua_estrangeira = [cn, ch, mt]

for prova in provas_sem_lingua_estrangeira:
    prova["TP_LINGUA"] = None

# Concatenação dos resultados

RESULTADO_PROVA = pd.concat([cn,ch,lc,mt], ignore_index = True)

# Conversão do tipo de dado de float para int da coluna CO_PROVA

RESULTADO_PROVA["CO_PROVA"] = RESULTADO_PROVA["CO_PROVA"].astype(int)

#### TABELA DE DADOS DAS PROVAS DO ENEM

In [4]:
import pandas as pd

# Leitura dos dados das planilhas dos itens das provas do ENEM

itens2019 = pd.read_csv(r"E:\fbd\microdados_enem_2019\DADOS\ITENS_PROVA_2019.csv", encoding = "latin1", sep = ";")
itens2020 = pd.read_csv(r"E:\fbd\microdados_enem_2020\DADOS\ITENS_PROVA_2020.csv", encoding = "latin1", sep = ";")
itens2021 = pd.read_csv(r"E:\fbd\microdados_enem_2021\DADOS\ITENS_PROVA_2021.csv", encoding = "latin1", sep = ";")
itens2022 = pd.read_csv(r"E:\fbd\microdados_enem_2022\DADOS\ITENS_PROVA_2022.csv", encoding = "latin1", sep = ";")
itens2023 = pd.read_csv(r"E:\fbd\microdados_enem_2023\microdados_enem_2023\DADOS\ITENS_PROVA_2023.csv", encoding = "latin1", sep = ";")

# Criação das tabelas das provas de cada ano do enem. 
# O processo envolve remoção de colunas irrelevantes, exclusão de duplicatas e reordenação das colunas

provas2019 = itens2019.iloc[:, [1,10,11]].drop_duplicates().reset_index(drop=True)
provas2019["ANO_ENEM"] = 2019
provas2019 = provas2019[["CO_PROVA", "SG_AREA", "TX_COR", "ANO_ENEM"]]

provas2020 = itens2020.iloc[:, [1,10,11]].drop_duplicates().reset_index(drop=True)
provas2020["ANO_ENEM"] = 2020
provas2020 = provas2020[["CO_PROVA", "SG_AREA", "TX_COR", "ANO_ENEM"]]

provas2021 = itens2021.iloc[:, [1,10,11]].drop_duplicates().reset_index(drop=True)
provas2021["ANO_ENEM"] = 2021
provas2021 = provas2021[["CO_PROVA", "SG_AREA", "TX_COR", "ANO_ENEM"]]

provas2022 = itens2022.iloc[:, [1,10,11]].drop_duplicates().reset_index(drop=True)
provas2022["ANO_ENEM"] = 2022
provas2022 = provas2022[["CO_PROVA", "SG_AREA", "TX_COR", "ANO_ENEM"]]

provas2023 = itens2023.iloc[:, [1,10,11]].drop_duplicates().reset_index(drop=True)
provas2023["ANO_ENEM"] = 2023
provas2023 = provas2023[["CO_PROVA", "SG_AREA", "TX_COR", "ANO_ENEM"]]

# Concatenar as 5 planilhas para formar a tabela PROVA

PROVA = pd.concat([provas2019,provas2020,provas2021,provas2022,provas2023], ignore_index = True)

# Padronizar o formato do dado da coluna referente à cor da prova

PROVA["TX_COR"] = PROVA["TX_COR"].str.title()

### LOAD

In [5]:
# Carregar dados nas tabelas ja criadas no banco MySQL

EDICAO_ENEM.to_sql(name = "edicao_enem", con = engine, if_exists = "append", index = False)

REGIAO.to_sql(name = "regiao", con = engine, if_exists = "append", index = False)

UNIDADE_FEDERATIVA.to_sql(name = "unidade_federativa", con = engine, if_exists = "append", index = False)

REGIAO_GEOGRAFICA_INTERMEDIARIA.to_sql(name = "regiao_geografica_intermediaria", con = engine, if_exists = "append", index = False)

REGIAO_GEOGRAFICA_IMEDIATA.to_sql(name = "regiao_geografica_imediata", con = engine, if_exists = "append", index = False)

MUNICIPIO.to_sql(name = "municipio", con = engine, if_exists = "append", index = False)

QUESTAO_QUESTIONARIO.to_sql(name = "questao_questionario", con = engine, if_exists = "append", index = False)

PROVA.to_sql(name = "prova", con = engine, if_exists = "append", index = False)

PARTICIPANTE.to_sql(name = "participante", con = engine, if_exists = "append", index = False)

RESPOSTA_QUESTIONARIO.to_sql(name = "resposta_questionario", con = engine, if_exists = "append", index = False)

RESULTADO_REDACAO.to_sql(name = "resultado_redacao", con = engine, if_exists = "append", index = False)

RESULTADO_PROVA.to_sql(name = "resultado_prova", con = engine, if_exists = "append", index = False)

108000